In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

pd.set_option("display.max_rows", 100)

path = Path("data")
neighbourhood_path = path.joinpath("neighbourhoods")
ttc_path = path.joinpath("ttc")
housing_path = path.joinpath("housing")

In [ ]:
import random
import time
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup


def fetch_sold_url():
    total_pages = 345
    links = pd.DataFrame(columns=["link"])
    curr_link = 0

    for page in range(1, total_pages + 1):
        user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) Gecko/20100101 Firefox/144.0"
        token = ""
        url = (
            "https://www.zolo.ca/index.php?s_r=3"
            if page == 1
            else f"https://www.zolo.ca/index.php/page-{page}?s_r=3"
        )
        headers = {"User-Agent": user_agent, "Cookie": f"token={token}"}

        request = Request(url, None, headers)
        raw = urlopen(request).read()
        soup = BeautifulSoup(raw, "html.parser")
        postings = soup.find_all("a", class_="address text-primary")

        for posting in postings:
            links.loc[curr_link] = str(posting.get("href"))
            curr_link += 1

        time.sleep(random.uniform(1, 4))
    links.to_csv(housing_path.joinpath("house_links.csv"), index=False)

In [ ]:
import os

import googlemaps


def normalize_feature(label):
    features = [
        "days_on_market",
        "sold_date",
        "sold_price",
        "age",
        "area",
        "municipality",
        "list_price",
        "bedrooms",
        "bedrooms_plus",
        "bathrooms",
        "kitchens",
        "rooms",
        "central_vac",
        "fireplace",
        "basement",
        "heating",
        "heating_type",
        "heating_fuel",
        "garage",
        "parking_places",
        "parking_total",
        "covered_parking_places",
        "address",
        "lat",
        "lon",
    ]
    feature_map = {
        "type": "house_type",
        "style": "house_style",
        "size (sq ft)": "size",
        "mls®#": "mls",
        "den/family room": "den_or_family_room",
        "air conditioning": "air_conditioning_type",
        "community": "neighbourhood",
    }
    temp_label = label.strip().lower()
    if temp_label in feature_map:
        return feature_map[temp_label]
    temp_label = "_".join(temp_label.split(" "))
    return temp_label if temp_label in features else None


def fetch_sold_details(houses, links):
    skipped = []

    for i in range(links.shape[0]):
        try:
            if i > 0 and i % 250 == 0:
                time.sleep(random.uniform(60, 120))

            url = links.at[i, "link"]
            user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) Gecko/20100101 Firefox/144.0"
            token = ""
            headers = {"User-Agent": user_agent, "Cookie": f"token={token}"}

            request = Request(url, None, headers)
            raw = urlopen(request).read()
            soup = BeautifulSoup(raw, "html.parser")

            # Look for all features
            for label_dt in soup.find_all("dt", class_="column-label"):
                label_text = label_dt.text.strip().lower()

                feature_name = normalize_feature(label_text)
                if feature_name is None:
                    continue

                span = label_dt.find_next_sibling("dd").find("span")
                if not span:
                    continue
                feature_value = span.text.strip().lower()
                houses.at[i, feature_name] = feature_value

            for label_div in soup.find_all("div", class_="column-label"):
                label_text = label_div.text.strip().lower()

                feature_name = normalize_feature(label_text)
                if feature_name is None:
                    continue

                span = label_div.find_next_sibling("div").find("span")
                if not span:
                    continue
                feature_value = span.text.strip().lower()

                if feature_name == "heating":
                    if (
                        pd.isna(houses.at[i, "heating"])
                        or houses.at[i, "heating"] == ""
                    ):
                        houses.at[i, "heating"] = feature_value
                    else:
                        houses.at[i, "heating"] += f", {feature_value}"
                else:
                    houses.at[i, feature_name] = feature_value

            # Special features not in the description
            address = soup.find("h1", class_="address").text.strip().lower()
            houses.at[i, "address"] = address

            print("Progress:", i)
            print(len(houses))
            time.sleep(random.uniform(1, 3))
        except Exception as e:
            print(f"Skipped {i}: {e}")
            skipped.append(i)
            continue

    print("Total skipped:", len(skipped))


def geocode_loc(houses):
    key = os.getenv("GOOGLE_API_KEY")
    gmaps = googlemaps.Client(key=key)

    for i, row in houses.iterrows():
        if not (pd.isna(row["lat"]) or pd.isna(row["lon"])):
            continue

        address = f'{row["address"]}, {row["neighbourhood"]}'
        geocode_result = gmaps.geocode(address)

        if geocode_result:
            loc = geocode_result[0]["geometry"]["location"]
            houses.at[i, "lat"] = loc["lat"]
            houses.at[i, "lon"] = loc["lng"]

        time.sleep(0.2)

In [ ]:
def split_heating(houses):
    split = houses["heating"].str.split(", ", n=1, expand=True)
    houses["heating_fuel"] = split[0]
    houses["heating_type"] = split[1]

    houses.drop(columns=["heating"], inplace=True)

# Neighbourhood Mapping


In [94]:
# Populating the table with housing data
links = pd.read_csv(housing_path.joinpath("house_links.csv"))

if (p := housing_path.joinpath("housing.csv")).exists():
    houses = pd.read_csv(p)
else:
    houses = pd.DataFrame(
        columns=[
            "days_on_market",
            "sold_date",
            "sold_price",
            "age",
            "house_type",
            "house_style",
            "size",
            "area",
            "municipality",
            "neighbourhood",
            "list_price",
            "bedrooms",
            "bedrooms_plus",
            "bathrooms",
            "kitchens",
            "rooms",
            "den_or_family_room",
            "central_vac",
            "air_conditioning_type",
            "fireplace",
            "basement",
            "heating",
            "heating_type",
            "heating_fuel",
            "garage",
            "parking_places",
            "parking_total",
            "covered_parking_places",
            "mls",
            "address",
            "lat",
            "lon",
        ]
    )

In [95]:
profile_path = neighbourhood_path.joinpath("neighbourhood-profiles-2021-158-model.xlsx")

profile = pd.read_excel(profile_path)
profile = profile.set_index(profile.columns[0]).T
profile.index = profile.index.str.strip()

profile_to_house_map = {
    "Cabbagetown-South St. James Town": "Cabbagetown-South St.James Town",
    "Danforth-East York": "Danforth East York",
    "East End Danforth": "East End-Danforth",
    "North St. James Town": "North St.James Town",
    "O`Connor Parkview": "O'Connor-Parkview",
    "Taylor Massey": "Taylor-Massey",
    "Yonge-St. Clair": "Yonge-St.Clair",
}

profile.index = profile.index.to_series().replace(profile_to_house_map)

In [96]:
import re

cols = []
for idx, col in enumerate(profile.columns):
    match = re.match("\\s+", col)
    indent = 0 if match is None else len(match.group())
    cols.append((idx, indent // 2, col[indent:]))

unique_names = []
current_path = {}

for idx, level, name in cols:
    if level == 0:
        current_path = {}
    current_path[level] = name
    full_name = " > ".join([current_path[i] for i in range(level + 1)])
    unique_names.append(full_name)

profile = profile.convert_dtypes()
profile.columns = unique_names

In [97]:
neighbourhoods = gpd.read_file(neighbourhood_path.joinpath("neighbourhoods.geojson"))

neighbourhoods = neighbourhoods[["AREA_NAME", "AREA_SHORT_CODE", "geometry"]]
neighbourhoods["AREA_NAME"] = neighbourhoods["AREA_NAME"].str.strip()
t_all_data = neighbourhoods.merge(
    profile, left_on="AREA_NAME", right_index=True, how="left"
).reset_index(drop=True)

t_all_data = t_all_data.rename(columns={"AREA_NAME": "neighbourhood"})
del profile

# Housing Statistics


- Neighbourhood Income (maybe household)
- Age (mean, median)
- Housings Prices projected on Neighbourhood Income
- Demographic (Language, Ethnicity)
- Population (Change, Total)
- Education
- Commuting (public transit access, commute time breakdown)
- Transit
- Listing


In [98]:
price_floor = houses["sold_price"].quantile(0.05)
houses = houses[houses["sold_price"] >= price_floor]

house_type_mapping = {
    "condo apartment": "Condo Apt",
    "co-op apartment": "Co-Op Apt",
    "co-ownership apartment": "Co-Ownership Apt",
    "detached": "Detached",
    "semi-detached": "Semi-Detached",
    "condo townhouse": "Condo Townhouse",
    "att/row/townhouse": "Att/Row/Townhouse",
    "common element condo": "Comm Element Condo",
    "link": "Link",
    "duplex": "Multiplex",
    "triplex": "Multiplex",
    "fourplex": "Multiplex",
    "multiplex": "Multiplex",
}

houses.loc[:, "final_type"] = houses["house_type"].map(house_type_mapping)
houses = houses.dropna(subset=["final_type"])
houses = houses.drop(columns=["house_type"])
houses = houses.rename(columns={"final_type": "house_type"})

In [99]:
# Moving neighbourhoods from 140 model to 158 (2021 onwards)
houses = gpd.GeoDataFrame(
    houses, geometry=gpd.points_from_xy(houses.lon, houses.lat), crs=neighbourhoods.crs
)

# Join houses (140-neighbourhood names) with 158-model neighbourhood names
houses = gpd.sjoin(
    houses,
    neighbourhoods[["AREA_NAME", "geometry"]],
    how="left",
    predicate="within",
)

del neighbourhoods

houses = houses.drop(
    columns=["municipality", "neighbourhood", "index_right"]
)
houses = houses.rename(columns={"AREA_NAME": "neighbourhood"})
houses.head(2)

,days_on_market,sold_date,sold_price,age,house_style,size,area,list_price,bedrooms,bedrooms_plus,...,parking_places,parking_total,covered_parking_places,mls,address,lat,lon,house_type,geometry,neighbourhood
0,21,2025-11-12,740000.0,no data,3-storey,1100-1500,toronto,699900,3.0,1.0,...,1.0,2.0,1.0,w12476861,127 torbarrie road,43.726329,-79.523197,Att/Row/Townhouse,POINT (-79.5232 43.72633),Oakdale-Beverley Heights
1,9,2025-11-12,980000.0,no data,2-storey,700-1100,toronto,989000,2.0,NaN,...,0.0,0.0,0.0,e12501558,99 newmarket avenue,43.689673,-79.304911,Semi-Detached,POINT (-79.30491 43.68967),East End-Danforth


In [34]:
houses.groupby("neighbourhood")["sold_price"].agg(["median", "count"]).sort_values(
    by="median", ascending=False
).tail(10)

,median,count
neighbourhood,,
Agincourt South-Malvern West,618500.0,86
Avondale,612500.0,104
Harbourfront-CityPlace,608000.0,213
Moss Park,594000.0,148
North Toronto,580500.0,82
Taylor-Massey,575000.0,26
Etobicoke City Centre,565000.0,173
Mount Olive-Silverstone-Jamestown,562500.0,52
Henry Farm,541250.0,98


In [13]:
from bokeh.io import output_file, output_notebook, show
from bokeh.models import (
    ColorBar,
    ColumnDataSource,
    GeoJSONDataSource,
    HoverTool,
    LinearColorMapper,
    NumeralTickFormatter,
)
from bokeh.palettes import Viridis
from bokeh.plotting import figure

In [ ]:
# Convert neighbourhood geopandas to json
geosource = GeoJSONDataSource(geojson=t_all_data.to_json())

p = figure(title="Toronto Neighbourhoods", height=600, width=900)
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

p.patches(
    "xs", "ys", source=geosource, line_color="white", line_width=0.25, fill_alpha=1
)

# Display figure in Jupyter
output_notebook()
show(p)

In [ ]:
geosource = GeoJSONDataSource(geojson=t_all_data.to_json())
income_col = "Total - Income statistics for private households - 25% sample data > Median total income of household in 2020 ($)"

p = figure(title="Toronto Neighbourhoods Median Income", height=600, width=900)
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

palette = Viridis[11][::-1]
color_mapper = LinearColorMapper(
    palette=palette,
    low=t_all_data[income_col].min(),
    high=t_all_data[income_col].max(),
)

hover = HoverTool(
    tooltips=[
        ("Neighbourhood", "@neighbourhood"),
        ("Median Income", f"@{{{income_col}}}{{($ 0,0)}}"),
    ]
)
p.add_tools(hover)

# Add patch renderer to figure
p.patches(
    "xs", "ys", source=geosource, line_color="white", line_width=0.5, fill_alpha=1
)

r = p.patches(
    "xs",
    "ys",
    source=geosource,
    line_color="white",
    line_width=0.5,
    fill_alpha=1,
    fill_color={
        "field": income_col,
        "transform": color_mapper,
    },
    muted_alpha=0,
    legend_label="Median Income",
)
color_bar = r.construct_color_bar(
    padding=1,
    width=600,
    location=(0, 0),
    formatter=NumeralTickFormatter(format="$0a"),
)
p.add_layout(color_bar, "below")

p.legend.click_policy = "mute"
p.legend.location = "top_right"

# Display figure in Jupyter
output_notebook()
show(p)

In [ ]:
geosource = GeoJSONDataSource(geojson=t_all_data.to_json())
avg_age_col = "Median age of the population"
med_age_col = "Average age of the population"

p = figure(title="Toronto Neighbourhoods Population Age", height=600, width=900)
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

palette = Viridis[11][::-1]

color_mapper = LinearColorMapper(
    palette=palette,
    low=t_all_data[[avg_age_col, med_age_col]].values.min(),
    high=t_all_data[[avg_age_col, med_age_col]].values.max(),
)

hover = HoverTool(
    tooltips=[
        ("Neighbourhood", "@neighbourhood"),
        ("Median age", f"@{{{med_age_col}}}{{0.0}} years"),
        ("Average age", f"@{{{avg_age_col}}}{{0.0}} years"),
    ]
)
p.add_tools(hover)

# Add patch renderer to figure
p.patches(
    "xs", "ys", source=geosource, line_color="white", line_width=0.5, fill_alpha=1
)

r = p.patches(
    "xs",
    "ys",
    source=geosource,
    line_color="white",
    line_width=0.5,
    fill_alpha=1,
    fill_color={
        "field": med_age_col,
        "transform": color_mapper,
    },
    muted_alpha=0,
    legend_label="Median Age",
)

p.patches(
    "xs",
    "ys",
    source=geosource,
    line_color="white",
    line_width=0.5,
    fill_alpha=1,
    fill_color={
        "field": avg_age_col,
        "transform": color_mapper,
    },
    muted_alpha=0,
    legend_label="Average Age",
)

color_bar = ColorBar(
    color_mapper=color_mapper,
    padding=5,
    width=600,
    location=(0, 0),
)
p.add_layout(color_bar, "below")

p.legend.click_policy = "mute"
p.legend.location = "top_right"

# Display figure in Jupyter
output_notebook()
show(p)

In [105]:
from bokeh.palettes import Plasma

geosource = GeoJSONDataSource(geojson=t_all_data.to_json())
house_geosource = GeoJSONDataSource(geojson=houses.to_json())

income_col = "Total - Income statistics for private households - 25% sample data > Median total income of household in 2020 ($)"

p = figure(title="Toronto Housing Projected on Median Income", height=600, width=900)
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = None

nb_palette = Viridis[11][::-1]
nb_color_mapper = LinearColorMapper(
    palette=nb_palette,
    low=t_all_data[income_col].min(),
    high=t_all_data[income_col].max(),
)

p.patches(
    "xs", "ys", source=geosource, line_color="white", line_width=0.5, fill_alpha=1
)

# Income glyph
nb_r = p.patches(
    "xs",
    "ys",
    source=geosource,
    line_color="white",
    line_width=0.5,
    fill_alpha=0.7,
    fill_color={
        "field": income_col,
        "transform": nb_color_mapper,
    },
    muted_alpha=0,
    legend_label="Median Income",
)

nb_color_bar = nb_r.construct_color_bar(
    padding=1,
    width=600,
    location=(0, 0),
    formatter=NumeralTickFormatter(format="$0a"),
)

p.add_layout(nb_color_bar, "below")

# House palette
house_palette = Plasma[11]
house_color_mapper = LinearColorMapper(
    palette=house_palette,
    low=houses["sold_price"].min(),
    high=houses["sold_price"].max(),
)

# Housing glyph
house_r = p.circle(
    "x",
    "y",
    color="orange",
    alpha=0.3,
    radius=0.0007,
    fill_color={"field": "sold_price", "transform": house_color_mapper},
    source=house_geosource,
)

house_color_bar = house_r.construct_color_bar(
    padding=1, width=600, location=(0, 0), formatter=NumeralTickFormatter(format="$0a")
)

p.add_layout(house_color_bar, "below")

hover = HoverTool(
    renderers=[house_r],
    tooltips=[
        ("Neighbourhood", "@neighbourhood"),
        ("Median Income", f"@{{{income_col}}}{{($ 0,0)}}"),
        ("Price", "$ @sold_price{0,0}"),
        ("Type", "@house_type"),
    ],
)
p.add_tools(hover)

p.legend.click_policy = "mute"
p.legend.location = "top_right"

# Display figure in Jupyter
# output_notebook()
show(p)

# Transit


In [ ]:
route_type = {
    "tram": 0,
    "streetcar": 0,
    "light_rail": 0,
    "subway": 1,
    "metro": 1,
    "rail": 2,
    "bus": 3,
    "ferry": 4,
    "cable_tram": 5,
    "aerial_lift": 6,
    "furnicular": 7,
    "trolleybus": 11,
    "monorail": 12,
}

dtype_ids = {"route_id": str, "trip_id": str, "stop_id": str}


def read_gtfs(file_name, dtype=None):
    if dtype is not None:
        return pd.read_csv(ttc_path.joinpath(file_name), dtype=dtype)
    return pd.read_csv(ttc_path.joinpath(file_name))

In [81]:
stops = read_gtfs("stops.txt", dtype=dtype_ids)
stops = stops[stops["stop_code"].str.isdigit()]
routes = read_gtfs("routes.txt", dtype=dtype_ids)
trips = read_gtfs("trips.txt", dtype=dtype_ids)
stop_times = read_gtfs("stop_times.txt", dtype=dtype_ids)
shapes = read_gtfs("shapes.txt")


subway_routes = routes[routes["route_type"] == route_type["subway"]]
rail_routes = routes[routes["route_type"] == route_type["rail"]]
bus_routes = routes[routes["route_type"] == route_type["bus"]]

subway_trips = trips[trips["route_id"].isin(subway_routes["route_id"])]
rail_trips = trips[trips["route_id"].isin(rail_routes["route_id"])]
bus_trips = trips[trips["route_id"].isin(bus_routes["route_id"])]

subway_stop_times = stop_times[stop_times["trip_id"].isin(subway_trips["trip_id"])]
rail_stop_times = stop_times[stop_times["trip_id"].isin(rail_trips["trip_id"])]
bus_stop_times = stop_times[stop_times["trip_id"].isin(bus_trips["trip_id"])]

subway_trips_shapes = (
    subway_trips[["route_id", "trip_id", "shape_id"]]
    .drop_duplicates("shape_id")
    .merge(subway_routes[["route_id", "route_color"]], on="route_id", how="left")
)


subway_stops = None

if not ttc_path.joinpath("subway_stops.csv").exists():
    subway_stops = (
        stops[stops["stop_id"].isin(subway_stop_times["stop_id"])]
        .drop_duplicates("stop_id")
        .assign(
            stop_name_clean=lambda df: df["stop_name"]
            .str.split(" - ", n=1)
            .str[0]
            .str.strip()
        )
        .drop_duplicates(subset=["stop_name_clean"])
    )
    subway_stops.to_csv(ttc_path.joinpath("subway_stops.csv"), index=False)
else:
    subway_stops = read_gtfs("subway_stops.csv", dtype=dtype_ids)

rail_stops = stops[stops["stop_id"].isin(rail_stop_times["stop_id"])].drop_duplicates(
    "stop_id"
)
bus_stops = stops[stops["stop_id"].isin(bus_stop_times["stop_id"])].drop_duplicates(
    "stop_id"
)

In [82]:
def get_transit_shape(trips_shapes, shape_df):
    used_shapes_id = trips_shapes["shape_id"].unique()
    used_shapes = shape_df[shape_df["shape_id"].isin(used_shapes_id)]
    rows = []

    for route_id in trips_shapes["route_id"].unique():
        line_shapes = trips_shapes[trips_shapes["route_id"] == route_id]
        shape_counts = (
            used_shapes[used_shapes["shape_id"].isin(line_shapes["shape_id"])]
            .groupby("shape_id")["shape_pt_sequence"]
            .count()
        )
        full_line_shape_id = shape_counts.idxmax()
        line_shape_pts = used_shapes[
            used_shapes["shape_id"] == full_line_shape_id
        ].sort_values("shape_pt_sequence")
        line_geom = LineString(line_shape_pts[["shape_pt_lon", "shape_pt_lat"]].values)
        route_color = f"#{line_shapes['route_color'].iloc[0]}"

        rows.append(
            {
                "route_id": route_id,
                "shape_id": full_line_shape_id,
                "geometry": line_geom,
                "route_color": route_color,
            }
        )
    return gpd.GeoDataFrame(rows, crs="EPSG:4326")